In [ ]:
### 합성곱 신경망 시각화


## 가중치 시각화 - 최적 CNN 모델의 경우

import keras

model = keras.models.load_model('sample_data/best-cnn-model.keras') # 08_2에서 생성한 최적 CNN 모델 로드

# 입력층 가중치/절편 다루기
model.layers
conv = model.layers[0] # 모델 0번째 층 => 입력층 객체 호출
print(conv.weights[0].shape, conv.weights[1].shape) # 입력층 가중치/절편 크기 출력
conv_weights = conv.weights[0].numpy() # 입력층 가중치를 numpy 배열로 변환
print(conv_weights.mean(), conv_weights.std()) # 입력층 가중치 평균/표준편차 출력

import matplotlib.pyplot as plt
plt.figure(figsize =(13,5))
plt.hist(conv_weights.reshape(-1, 1)) # 입력층 가중치를 1차원 배열로 변환
plt.xlabel('weight')
plt.ylabel('count')
plt.show()

# 입력층 커널 32개에 의한 가중치 맵 32개 시각화

fig, axs = plt.subplots(2, 16, figsize=(15, 2))

for i in range(2):
  for j in range(16):
    axs[i, j].imshow(conv_weights[:, :, :, i * 16 + j], vmin = -0.5, vmax = 0.5) #[높이, 너비, 입력 채널, 커널]
    axs[i, j].axis('off')
plt.show()



## 가중치 시각화 - 훈련하지 않은 CNN 모델의 경우

no_training_model = keras.Sequential()
no_training_model.add(keras.layers.Input(shape = (28,28,1)))
no_training_model.add(keras.layers.Conv2D(32, kernel_size = 3, activation = 'relu', padding = 'same'))

# 최적 CNN 모델의 경우와 마찬가지로 입력층 호출 -> 입력층 가중치 시각화
no_training_conv = no_training_model.layers[0]

print(no_training_conv.weights[0].shape)

no_training_conv_weights = no_training_conv.weights[0].numpy() # 입력층 가중치 Numpy 변환
print(no_training_conv_weights.mean(), no_training_conv_weights.std()) # 가중치 평균 & 표준편차 도출

import matplotlib.pyplot as plt
plt.figure(figsize =(13,5))
plt.hist(no_training_conv_weights.reshape(-1, 1)) # 입력층 가중치를 1차원 배열로 변환
plt.xlabel('weight')
plt.ylabel('count')
plt.show() # 최적 CNN 모델보다 가중치 분포가 고르다(evenly)

fig, axs = plt.subplots(2, 16, figsize=(15, 2))

for i in range(2):
  for j in range(16):
    axs[i, j].imshow(no_training_conv_weights[:, :, :, i * 16 + j], vmin = -0.5, vmax = 0.5) #[높이, 너비, 입력 채널, 커널]
    axs[i, j].axis('off')
plt.show()

# 결론: CNN 모델 초기 가중치는 균등(랜덤)하게 초기화
# -> 이후 훈련을 거치며 특정 패턴에 비중을 두며 업데이트



## 함수형 API

inputs = keras.Input(shape = (28,28,1))
dense1 = keras.layers.Dense(100, activation = 'relu')
dense2 = keras.layers.Dense(10, activation = 'softmax')

hidden = dense1(inputs) # 입력층-은닉층 체인
outputs = dense2(hidden) # [입력층 - 은닉층] - 출력층 체인

functional_model = keras.Model(inputs, outputs)
print(model.inputs)



## 함수형 API를 활용한 첫번째 합성곱층 가중치 시각화
conv_acti = keras.Model(model.inputs, model.layers[0].output) # 함수형 API, 입력층-첫번째 합성곱층 모델 형성

(train_input, train_target), (test_input, test_target) = keras.datasets.fashion_mnist.load_data()

plt.imshow(train_input[0], cmap = 'gray_r')
plt.show()

train_input = train_input.reshape(-1, 28, 28, 1) / 255.0

ankle_boot = train_input[0:1].reshape(-1, 28, 28, 1) / 255.0

feature_maps = conv_acti.predict(ankle_boot) # 입력층-합성곱층 모델로 부츠 이미지 에측

print(feature_maps.shape)

fig, axs = plt.subplots(4, 8, figsize=(15, 8))

for i in range(4):
  for j in range(8):
    axs[i, j].imshow(feature_maps[0, :, :, i * 8 + j])
    axs[i, j].axis('off')
plt.show()



## 함수형 API를 활용한 두번째 합성곱층 가중치 시각화
conv2_acti = keras.Model(model.inputs, model.layers[2].output) # 함수형 API, 입력층-두번째 합성곱층 모델 형성
feature_maps = conv2_acti.predict(ankle_boot) # 입력층-합성곱층 모델로 부츠 이미지 에측

print(feature_maps.shape)

fig, axs = plt.subplots(8, 8, figsize=(12, 12))

for i in range(8):
  for j in range(8):
    axs[i, j].imshow(feature_maps[0, :, :, i * 8 + j])
    # feature_maps 인수 순서: (배치, 높이, 너비, 채널) <-> 가중치(높이, 너비, 채널, 커널)
    axs[i, j].axis('off')
plt.show()

In [ ]:
### 합성곱 가중치(커널) 오버래핑 실습

# 최적 CNN 모델 세팅
import keras
import matplotlib.pyplot as plt # Added matplotlib import

model = keras.models.load_model('sample_data/best-cnn-model.keras') # 08_2에서 생성한 최적 CNN 모델 로드



# 합성곱층 특성맵을 추출하기 위해 model의 '입력층'-'첫번째 합성곱층'으로 구성된 CNN 모델 생성
conv_acti = keras.Model(model.inputs, model.layers[0].output)

# 데이터 추출 & 전처리 & 이미 target을 알고 있는 0번째 훈련 샘플 전처리(ankle_boot)
(train_input, train_target), (test_input, test_target) = keras.datasets.fashion_mnist.load_data()

train_input = train_input.reshape(-1, 28, 28, 1) / 255.0

ankle_boot = train_input[0:1].reshape(-1, 28, 28, 1) / 255.0

# '입력층'-'첫번째 합성곱층'으로 구성된 CNN 모델의 'ankle_boot' 입력 예측 특성맵 저장
feature_maps = conv_acti.predict(ankle_boot)

# CNN 모델의 특성맵 평균값 & 최댓값 저장(활성화된 이미지 시각화에 사용)
overlay_mean = feature_maps.mean(axis=-1)
overlay_max  = feature_maps.max(axis=-1)

# CNN 모델 특성맵 평균 & 최댓값 시각화 - 32개 필터 결과층을 합한 것과 유사한 결과 도출
plt.imshow(overlay_mean[0], cmap="gray"); plt.axis("off"); plt.title("mean over channels"); plt.show()
plt.imshow(overlay_max[0], cmap="gray");  plt.axis("off"); plt.title("max over channels");  plt.show()